# 🚀 Personal Expense Tracker

**Lumexa Python Foundations · Course 2: Problem Solving & Program Design · Lesson 7**

| | |
|---|---|
| **Project** | Personal Expense Tracker (portfolio project 5 of 9) |
| **You will practise** | records, JSON files, `try` / `except`, input validation, functions, dictionaries, CSV export |
| **Time** | about 60 minutes |

Build a program that records, retrieves, validates and organises your spending, and **remembers everything** after the program stops, because the data is saved to a file.

## 🧭 How to use this notebook

1. **Keep your own copy.** Choose **File → Save a copy in Drive** first, so your work is saved in *your* Google Drive.
2. **Run the cells in order**, from the top, with **Shift + Enter** (or **Runtime → Run all**). Read the text above each cell before you run it.
3. Cells marked **✏️ Your turn** are for you to change or complete. They are safe to run before you finish them.
4. If something goes wrong, use **Runtime → Restart session and run all**. A notebook you can run from the top is a notebook you can trust.
5. **Demo mode:** by default the notebook "types" the answers for you, so *Run all* works without stopping. To play yourself, set `INTERACTIVE = True` in the setup cell and run it again.

> 💾 The tracker saves its data in a folder called `expense_tracker_data`. In Colab this folder lives in your session: it is kept while the notebook is open, and removed when the session ends. (On your own computer it stays for good.)

In [ ]:
# ⚙️ Setup — run this cell first
# INTERACTIVE = False -> demo mode: the notebook "types" scripted answers, so Run all works.
# INTERACTIVE = True  -> you type the answers yourself.
INTERACTIVE = False

demo_script = iter([])      # the scripted answers for demo mode (each demo cell fills this)

def ask(prompt):
    """Works like input(). In demo mode the next scripted answer is 'typed' for you."""
    if INTERACTIVE:
        return input(prompt)
    answer = next(demo_script, "")
    print(f"{prompt}{answer}")
    return answer

import csv
import json
from datetime import datetime
from pathlib import Path

CURRENCY = "৳"                      # change it to your own currency
MONTHLY_BUDGET = 1500               # the warning limit for one month

DATA_FOLDER = Path("expense_tracker_data")
DATA_FOLDER.mkdir(exist_ok=True)
EXPENSES_FILE = DATA_FOLDER / "expenses.json"
CSV_FILE = DATA_FOLDER / "expenses.csv"

print("Setup complete. Interactive mode is", "ON" if INTERACTIVE else "OFF (demo mode)")
print("Your data will be saved in the folder:", DATA_FOLDER)

## Part 1 · The data model

Every expense is a **record**: a dictionary with the same four keys. All the expenses together are a **list of dictionaries**.

> Dates are stored as text in the form `YYYY-MM-DD` (year-month-day). This form has a handy property: it sorts correctly, and `"2030-07-14".startswith("2030-07")` tells you the month.

In [ ]:
example = {"date": "2030-07-03", "category": "Food", "description": "Lunch at school", "amount": 120.0}
expenses = [example]

print(example["description"], "cost", CURRENCY, example["amount"])
print("Number of expenses:", len(expenses))

## Part 2 · Saving and loading (persistence)

To remember data after the program stops we use the **load → change → save** pattern with a JSON file.

`load_expenses()` must cope with two situations: the very **first run** (no file yet) and a **damaged file**. Both are handled with `try` and `except`, and in both cases we simply start with an empty list instead of crashing.

In [ ]:
def load_expenses(path=EXPENSES_FILE):
    """Return the saved list of expenses (an empty list on the first run or if the file is damaged)."""
    try:
        with open(path) as file:
            return json.load(file)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        print("⚠️  The data file is damaged. Starting with an empty list.")
        return []

def save_expenses(expenses, path=EXPENSES_FILE):
    """Write the whole list to the data file."""
    with open(path, "w") as file:
        json.dump(expenses, file, indent=2)

EXPENSES_FILE.unlink(missing_ok=True)          # start the demo from a clean slate
print("First run:", load_expenses())

# What if the file is damaged?
broken = DATA_FOLDER / "broken.json"
broken.write_text("{this is not valid json")
print("Damaged file:", load_expenses(broken))
broken.unlink()

## Part 3 · Validation: never trust what the user types

Each helper **keeps asking until the answer is valid**, so the rest of the program can rely on the data being right. Watch the scripted player make three mistakes: an impossible date, text instead of a number, and a negative amount.

In [ ]:
def read_date(prompt):
    """Ask for a date in the form YYYY-MM-DD until it is real."""
    while True:
        text = ask(prompt).strip()
        try:
            datetime.strptime(text, "%Y-%m-%d")     # raises ValueError for 2030-13-45
            return text
        except ValueError:
            print("  ⚠️  Please use the form YYYY-MM-DD, for example 2030-07-14.")

def read_amount(prompt):
    """Ask for a positive number."""
    while True:
        text = ask(prompt).strip()
        try:
            amount = float(text)
        except ValueError:
            print("  ⚠️  Please type a number, for example 12.50.")
            continue
        if amount <= 0:
            print("  ⚠️  The amount must be more than zero.")
            continue
        return round(amount, 2)

def read_text(prompt):
    """Ask for text that is not empty."""
    while True:
        text = ask(prompt).strip()
        if text:
            return text
        print("  ⚠️  This cannot be empty.")

demo_script = iter(["2030-13-45", "2030-07-03", "abc", "-5", "120"])
print("Date accepted:", read_date("Date: "))
print("Amount accepted:", read_amount("Amount: "))

## Part 4 · Adding an expense

`add_expense()` asks for the four details, builds a record, adds it to the list and **saves immediately**, so nothing is lost if the program stops.

We will add six expenses. (In demo mode the answers come from a prepared list; the notebook builds the script for us.)

In [ ]:
def add_expense(expenses):
    """Ask for the details of one expense and add it to the list."""
    expense = {
        "date": read_date("Date (YYYY-MM-DD): "),
        "category": read_text("Category: ").title(),
        "description": read_text("Description: "),
        "amount": read_amount(f"Amount ({CURRENCY}): "),
    }
    expenses.append(expense)
    save_expenses(expenses)
    print(f"  ✅ Saved: {expense['description']} ({CURRENCY}{expense['amount']:,.2f})")

sample_data = [
    ("2030-07-03", "food", "Lunch at the school canteen", "120"),
    ("2030-07-05", "transport", "Monthly bus pass", "300"),
    ("2030-07-09", "fun", "Cinema ticket", "450"),
    ("2030-07-14", "food", "Groceries for the week", "850"),
    ("2030-08-01", "books", "Python notebook", "180"),
    ("2030-08-02", "food", "Pizza night", "600"),
]
demo_script = iter([answer for row in sample_data for answer in row])

expenses = load_expenses()
for number in range(len(sample_data)):
    add_expense(expenses)
print("\nTotal expenses recorded:", len(expenses))

## Part 5 · Showing, totalling and filtering

- `show_expenses()` prints a neat numbered table.
- `total_by_category()` builds a **dictionary** of totals with the `.get(key, 0)` pattern from Lesson 4.
- `show_totals()` sorts the totals from largest to smallest and draws a simple text bar chart.
- `filter_by_month()` keeps only the expenses whose date starts with the month.
- `check_budget()` warns you when a month goes over the budget.

In [ ]:
def show_expenses(expenses, title="All expenses"):
    print(f"\n{title}")
    print("-" * 62)
    total = 0
    for number, e in enumerate(expenses, start=1):
        print(f"{number:>2}. {e['date']}  {e['category']:<10} {e['description']:<28} {CURRENCY}{e['amount']:>8,.2f}")
        total += e["amount"]
    print("-" * 62)
    print(f"{'Total':>48} {CURRENCY}{total:>8,.2f}")

def total_by_category(expenses):
    totals = {}
    for e in expenses:
        totals[e["category"]] = totals.get(e["category"], 0) + e["amount"]
    return totals

def amount_of(pair):
    return pair[1]                      # used to sort (category, total) pairs by the total

def show_totals(expenses):
    print("\nSpending by category (largest first)")
    for category, total in sorted(total_by_category(expenses).items(), key=amount_of, reverse=True):
        bar = "█" * int(total // 50)
        print(f"{category:<10} {CURRENCY}{total:>8,.2f}  {bar}")

def filter_by_month(expenses, month):
    """month looks like '2030-07'."""
    return [e for e in expenses if e["date"].startswith(month)]

def check_budget(expenses, month):
    total = sum(e["amount"] for e in filter_by_month(expenses, month))
    if total > MONTHLY_BUDGET:
        print(f"⚠️  {month}: you spent {CURRENCY}{total:,.2f}, which is {CURRENCY}{total - MONTHLY_BUDGET:,.2f} OVER your budget of {CURRENCY}{MONTHLY_BUDGET:,}.")
    else:
        print(f"✅ {month}: {CURRENCY}{total:,.2f} spent, within your budget of {CURRENCY}{MONTHLY_BUDGET:,}.")

show_expenses(expenses)
show_totals(expenses)
show_expenses(filter_by_month(expenses, "2030-07"), "July 2030")
check_budget(expenses, "2030-07")
check_budget(expenses, "2030-08")

## Part 6 · Proof that it remembers, and a CSV export

We **throw away** the list in memory (as if the program had been closed) and load it again from the file. Then we export the data to a **CSV** file, which a spreadsheet program can open.

In [ ]:
expenses = []                              # "close the program": everything in memory is gone
print("In memory now:", len(expenses), "expenses")

expenses = load_expenses()                 # "open the program again"
print("Loaded from the file:", len(expenses), "expenses  ✅")

def export_csv(expenses, path=CSV_FILE):
    with open(path, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["date", "category", "description", "amount"])
        writer.writeheader()
        writer.writerows(expenses)
    print("Exported", len(expenses), "expenses to", path)

export_csv(expenses)
print("\nFirst three lines of the CSV file:")
with open(CSV_FILE) as file:
    for line_number, line in enumerate(file):
        if line_number == 3:
            break
        print("  ", line.strip())

## Part 7 · The menu: everything together

A menu loop lets the user choose what to do until they quit. Because every task is its own function, the loop is short and easy to read. In demo mode it plays a scripted session.

In [ ]:
MENU = """
===== PERSONAL EXPENSE TRACKER =====
1. Add an expense
2. Show all expenses
3. Totals by category
4. Show one month
5. Export to CSV
6. Quit"""

def run_menu(expenses):
    while True:
        print(MENU)
        choice = ask("Choose 1-6: ").strip()
        if choice == "1":
            add_expense(expenses)
        elif choice == "2":
            show_expenses(expenses)
        elif choice == "3":
            show_totals(expenses)
        elif choice == "4":
            month = read_text("Month (YYYY-MM): ")
            show_expenses(filter_by_month(expenses, month), f"Expenses in {month}")
            check_budget(expenses, month)
        elif choice == "5":
            export_csv(expenses)
        elif choice == "6":
            print("Goodbye! Your data is saved. 👋")
            break
        else:
            print("  ⚠️  Please choose a number from 1 to 6.")

demo_script = iter(["9", "1", "2030-08-15", "fun", "Space museum ticket", "250",
                    "4", "2030-08", "3", "6"])
run_menu(expenses)

## ✏️ Your turn · Delete an expense

Complete `delete_expense()`. It receives the list and the **number** shown in the table (starting from 1). It must:

- refuse a number that is too small or too big (return `False`),
- otherwise remove that expense, save the file and return `True`.

Hint: `del expenses[number - 1]` removes an item from a list.

In [ ]:
def delete_expense(expenses, number):
    return None   # <- write your code here, then run the check below

test = [{"date": "2030-01-01", "category": "Test", "description": "A", "amount": 1.0},
        {"date": "2030-01-02", "category": "Test", "description": "B", "amount": 2.0}]
ok = delete_expense(test, 1) is True and len(test) == 1 and test[0]["description"] == "B"
refused = delete_expense(test, 9) is False and len(test) == 1
print("✅ delete_expense works!" if ok and refused else "⬜ delete_expense is not finished yet")

## 🏆 Stretch challenges

1. Add an **edit** option that changes the amount or category of an existing expense.
2. Add a **search** that finds expenses whose description contains a word.
3. Put the monthly budget in a **settings file** (`settings.json`) that is loaded when the program starts.
4. **Import** expenses from a CSV file, skipping (and reporting) any row with a bad date or amount.
5. Write a **monthly report** to a text file with the totals for each category.

## 🪞 Reflection and testing

Test your tracker with these edge cases and write down the result of each: a missing data file, an empty file, a negative amount, a badly formatted date, a very large amount, an empty category.

*Your test results:*

## 📤 How to submit

**File → Save a copy in Drive**, then **Share → Anyone with the link can view**, and send the link to your teacher. Make sure the notebook runs from the top with **Runtime → Restart session and run all**.